# HP-4 · LoRA cho dịch máy đẳng thời (NLLB-200)

**Ý tưởng.** Dịch máy thường chỉ tối ưu *đúng nghĩa*. Lồng tiếng còn cần *đúng độ
dài*: câu tiếng Việt phải đọc vừa khung thời gian của câu gốc. Ở đây ta dạy
NLLB-200 nhận thêm một **token ngân sách** ở đầu câu nguồn — `<len:12>` nghĩa là
"hãy dịch sao cho đọc khoảng 12 âm tiết" — rồi fine-tune bằng LoRA.

**Vì sao LoRA.** NLLB-200 distilled 600M có ~600 triệu tham số. LoRA chỉ huấn
luyện vài triệu tham số thêm vào, vừa một GPU phổ thông, và giữ nguyên mô hình gốc.

**Trạng thái: ĐÃ CHẠY (11/11/2026) cho cặp Trung → Việt.** Huấn luyện thật không chạy
trong notebook mà bằng script dòng lệnh, vì cần chạy tiếp được từ checkpoint và một
môi trường riêng có torch CUDA:

- `training/prepare_opus.py` — lọc OPUS OpenSubtitles vi–zh_CN (13,5 triệu cặp → 7,9 triệu đạt chuẩn → lấy mẫu 300.000)
- `training/build_corpus.py --extra … --holdout …` — gộp, loại tập kiểm thử
- `training/train_nllb_lora.py` — huấn luyện (RTX 4060 Laptop 8 GB: 1 epoch, 9.329 bước, 54 phút)
- `training/eval_nllb_lora.py` — đo trên bộ gold và tập dev

Notebook này giữ lại để **giải thích ý tưởng và dịch thử**; bảng số đo ở cuối và trong
`training/README.md`. Các ô huấn luyện vẫn có khoá `CHO_PHEP_CHAY`.

**Nhóm A (huấn luyện).** Đề cương: Anh, Trung, Hàn, Nhật → Việt. **Mới làm: Trung → Việt.**
**Nhóm B (chỉ kiểm tra zero-shot).** Pháp, Nga, Thái → Việt. Đây là giả thuyết H1
của đề cương: ràng buộc độ dài học được có lan sang ngôn ngữ chưa huấn luyện không. **Chưa đo.**

In [ ]:
# Chạy được dù mở notebook từ thư mục nào: đi ngược lên tới thư mục có backend/
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "backend").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Gốc dự án:", ROOT)

## Kiểm tra môi trường

In [ ]:
import importlib.util

need = {
    "torch": "--extra offline (hoặc --extra clone)",
    "transformers": "--extra offline",
    "peft": "uv pip install peft",
    "datasets": "uv pip install datasets",
}
missing = [n for n in need if importlib.util.find_spec(n) is None]

for name, how in need.items():
    mark = "✓" if name not in missing else "✗"
    print(f"  {mark} {name:<14} {'' if name not in missing else how}")

if missing:
    print("\nCòn thiếu:", ", ".join(missing))
else:
    import torch
    print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — LoRA sẽ rất chậm")

## Chuẩn bị dữ liệu

Mỗi mẫu gồm câu nguồn, câu đích tiếng Việt, và **ngân sách âm tiết** tính từ
chính câu đích trong tập huấn luyện. Lúc dùng thật thì ngân sách lấy từ khung
thời gian của cue chia cho tốc độ đọc (xem `backend/core/budget.py`).

In [ ]:
from backend.core import duration, isochron

# Cách gắn token nằm trong backend/core/isochron.py để script huấn luyện và app dùng chung
to_example = isochron.to_example

demo = [
    to_example("我們獲勝了", "Chúng ta đã thắng rồi"),
    to_example("I am the Cyclops and I will eat you all", "Ta là gã khổng lồ một mắt, ta sẽ ăn thịt các ngươi"),
]
for d in demo:
    print(d["src"])
    print("  ->", d["tgt"], f'({duration.syllables(d["tgt"])} âm tiết)')

In [ ]:
# Nạp corpus song ngữ thật. Định dạng mong đợi: JSONL mỗi dòng {"src": ..., "tgt": ...}
# HP-6 của đề cương sinh ra file này từ phụ đề OpenSubtitles.
CORPUS = ROOT / "data" / "corpus" / "vidub_train.jsonl"

if CORPUS.exists():
    import json

    pairs = [json.loads(ln) for ln in CORPUS.read_text(encoding="utf-8").splitlines() if ln.strip()]
    examples = [to_example(p["src"], p["tgt"]) for p in pairs]
    print(f"{len(examples)} mẫu từ {CORPUS}")
else:
    examples = demo
    print(f"Chưa có {CORPUS} — dùng tạm {len(demo)} mẫu minh hoạ.")
    print("HP-6 (dựng bộ dữ liệu ViDub) phải xong trước bước này.")

## Cấu hình LoRA

In [ ]:
from backend.config import MODEL_DIR

BASE_MODEL = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR = isochron.adapter_dir(MODEL_DIR)   # data/models/nllb_lora_dangthoi

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"], task_type="SEQ_2_SEQ_LM")
# Cấu hình đã chạy thật trên GPU 8 GB: batch 16 × tích luỹ 2, mô hình gốc để half precision
TRAIN = dict(learning_rate=2e-4, num_train_epochs=1,
             per_device_train_batch_size=16, gradient_accumulation_steps=2)

print("Mô hình gốc:", BASE_MODEL)
print("LoRA        :", LORA)
print("Huấn luyện  :", TRAIN)
print("Lưu vào     :", OUTPUT_DIR, "(đã có)" if isochron.has_adapter(MODEL_DIR) else "(chưa huấn luyện)")

## Huấn luyện

Chạy thật bằng dòng lệnh (dừng êm sau `--max-minutes`, chạy lại là tiếp tục từ checkpoint):

```bash
data/_train_venv/Scripts/python training/train_nllb_lora.py --origins extra --dev 500
```

Ô dưới đây là bản rút gọn để đọc hiểu, gọi đúng các hàm của script. Hai điều notebook
bản đầu làm thiếu và chạy thật mới lộ ra:

1. `add_tokens` tạo 60 hàng embedding **mới, ngẫu nhiên**; LoRA trên `q_proj/v_proj` không
   đụng tới embedding nên phải cho huấn luyện riêng 60 hàng đó (`trainable_token_indices`).
2. Batch 32 ở fp32 chiếm hơn 8 GB → Windows tràn sang RAM dùng chung, tốc độ rơi còn
   một nửa. Để mô hình gốc ở half precision, batch 16 × tích luỹ 2 thì vừa (≈ 92 mẫu/giây).

⚠ Đổi `CHO_PHEP_CHAY = True` sẽ huấn luyện lại ~1 giờ trên GPU.

In [ ]:
CHO_PHEP_CHAY = False   # đổi thành True để huấn luyện lại (~1 giờ trên RTX 4060)

if not CHO_PHEP_CHAY:
    print("Đang khoá. Đặt CHO_PHEP_CHAY = True để chạy thật.")
elif missing:
    print("Thiếu thư viện:", ", ".join(missing))
else:
    import subprocess

    # chạy đúng script dòng lệnh, để notebook và bản chạy thật không lệch nhau
    subprocess.run([sys.executable, str(ROOT / "training" / "train_nllb_lora.py"),
                    "--origins", "extra", "--dev", "500"], check=True)

## Dịch thử với ngân sách khác nhau\n\nCùng một câu, ép ngân sách ngắn dần — câu dịch phải ngắn theo.

In [ ]:
def dich_voi_ngan_sach(text_nguon: str, n_am_tiet: int | None) -> str:
    """Dịch một câu tiếng Trung, ép độ dài về khoảng `n_am_tiet` âm tiết (None = dịch tự do)."""
    if not isochron.has_adapter(MODEL_DIR):
        return "(chưa có mô hình đã huấn luyện)"
    from backend.core import translate

    # đúng hàm app gọi: tự nạp NLLB gốc + adapter khi thư mục adapter tồn tại
    return translate._translate_nllb([text_nguon], "zh", "vi", budgets=[n_am_tiet])[0]


for n in (None, 6, 10, 16):
    print(f"<len:{n}> ->", dich_voi_ngan_sach("我是独眼巨人，我要把你们都吃了", n))

## Đánh giá

Hai chỉ số đi cùng nhau, vì siết độ dài mà hỏng nghĩa thì vô nghĩa:

1. **BLEU / chrF** — nghĩa có giữ được không (`data/_bench/score_vi.py`).
2. **Tỉ lệ câu vượt khung** — đo bằng chính `budget.assess()` mà bản chạy thật dùng.

```bash
data/_train_venv/Scripts/python training/eval_nllb_lora.py
```

**Bộ gold** — 271 câu thoại phim cổ trang, dịch từng dòng, 68 đơn vị đọc:

| Bản dịch | BLEU | chrF | Vượt khung (đơn vị đọc) | Vượt khung (dòng) |
|---|---|---|---|---|
| NLLB gốc | 5,0 | 22,1 | 6/68 = 8,8 % | 67/271 |
| NLLB gốc + luật rút gọn (HP-5) | 5,0 | 22,1 | 5/68 = 7,4 % | 66/271 |
| NLLB + LoRA, không token | 5,6 | 22,3 | 6/68 = 8,8 % | 71/271 |
| NLLB + LoRA, dịch lại từng dòng với `<len:n>` | 5,9 | 22,0 | **0/68** | 56/271 |
| NLLB + LoRA, bộ siết của app (dịch lại cả đơn vị) | 5,5 | 20,5 | **0/68** | 51/271 |
| *Google (mốc)* | *7,9* | *28,9* | | |
| *LLM có ngữ cảnh (mốc)* | *41,6* | *63,3* | | |

**Tập dev** — 500 cặp OpenSubtitles không tham gia huấn luyện, n = số âm tiết của bản tham chiếu:

| Bản dịch | BLEU | chrF | Lệch trung bình (âm tiết) | Trong ±1 âm tiết |
|---|---|---|---|---|
| NLLB gốc | 12,0 | 30,2 | 2,06 | 50,4 % |
| NLLB + LoRA, không token | 13,3 | 30,9 | 1,90 | 55,2 % |
| NLLB + LoRA, `<len:n>` | 13,0 | 31,0 | **1,01** | **81,6 %** |
| NLLB + LoRA, `<len:0,7n>` (ép ngắn 30 %) | 12,1 | 28,8 | 1,23 | 73,4 % |

**Đọc bảng cho đúng.**

- Cái LoRA làm được là **điều khiển độ dài**: bám ngân sách ±1 âm tiết từ 50 % lên 82 %, và
  trên bộ gold hết hẳn đơn vị đọc vượt khung (bộ luật HP-5 chỉ gỡ được 1/6).
- **Chất lượng dịch gần như không đổi.** BLEU bộ gold 5,0 → 5,6 chưa có ý nghĩa thống kê
  (bootstrap ghép cặp 1.000 lượt: LoRA thắng 74 % số lượt), chrF đứng yên. NLLB dù có LoRA
  vẫn kém Google và kém xa LLM trên phim cổ trang: OpenSubtitles phần lớn là phim Âu–Mỹ,
  không dạy được "lang quân / thiếp / tại hạ".
- Ép ngắn 30 % thì mô hình chỉ ngắn đi được một phần (dài/n = 1,16) và mất ~2 điểm chrF —
  ngân sách là "mong muốn", không phải ràng buộc cứng.
- Bộ siết của app dịch lại cả đơn vị đọc rồi chia lại theo tỉ lệ ký tự nên điểm theo dòng
  thấp hơn (chrF 20,5) dù cũng hết vượt khung.
- Chưa làm: Anh/Hàn/Nhật → Việt, và kiểm tra zero-shot nhóm B (giả thuyết H1).

In [ ]:
from backend.core import budget, metrics

print("BLEU:", "có sacrebleu" if metrics.bleu(["a"], ["a"]) is not None else "cần: uv pip install sacrebleu")
print("Ngưỡng vượt khung dùng chung:", budget.TOLERANCE)

ket_qua = isochron.adapter_dir(MODEL_DIR) / "eval_gold.json"
if ket_qua.exists():
    import json

    for r in json.loads(ket_qua.read_text(encoding="utf-8"))["gold"]:
        print(f"{r['name']:16s} BLEU {r['bleu']:5.1f}  chrF {r['chrf']:5.1f}  vượt khung {r['over_units']}/{r['units']}")
else:
    print("Chưa có số đo — chạy training/eval_nllb_lora.py")